# 05 — Random Forest Demand Forecasting & RL Simulation
**AI-Driven Waste Collection & Route Optimization**

- Random Forest: predicts waste volume by zone and time-of-week
- Reinforcement Learning: simulates reward-based dynamic routing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sensor_df = pd.read_csv('../data/sensor_readings.csv', parse_dates=['timestamp'])
print('Data loaded.')

## Part A — Random Forest Demand Forecasting

In [ ]:
# Aggregate to zone-hourly demand
sensor_df['hour'] = sensor_df['timestamp'].dt.hour
sensor_df['dow']  = sensor_df['timestamp'].dt.dayofweek
sensor_df['week'] = sensor_df['timestamp'].dt.isocalendar().week.astype(int)
sensor_df['month']= sensor_df['timestamp'].dt.month

zone_hourly = sensor_df.groupby(['zone_type','hour','dow','week']).agg(
    avg_fill = ('fill_level_pct','mean'),
    max_fill = ('fill_level_pct','max'),
    num_bins = ('bin_id','nunique'),
).reset_index()

le = LabelEncoder()
zone_hourly['zone_enc'] = le.fit_transform(zone_hourly['zone_type'])

feature_cols = ['zone_enc','hour','dow','week']
X = zone_hourly[feature_cols]
y = zone_hourly['avg_fill']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_split=4,
                           n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
preds = rf.predict(X_te)

mae = mean_absolute_error(y_te, preds)
r2  = r2_score(y_te, preds)
cv  = cross_val_score(rf, X, y, cv=5, scoring='r2').mean()

print(f'Random Forest MAE  : {mae:.2f}%')
print(f'Random Forest R²   : {r2:.4f}')
print(f'Cross-val R² (5-fold): {cv:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Random Forest Demand Forecasting', fontsize=13, fontweight='bold')

axes[0].scatter(y_te, preds, alpha=0.4, s=10, color='#2196F3')
axes[0].plot([0,100],[0,100],'r--',linewidth=1.5)
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})')
axes[0].set_xlabel('Actual Avg Fill (%)')
axes[0].set_ylabel('Predicted Avg Fill (%)')
axes[0].grid(True, alpha=0.3)

feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
axes[1].barh(feat_imp.index, feat_imp.values, color='#4CAF50', alpha=0.8)
axes[1].set_title('Feature Importance')
axes[1].set_xlabel('Importance')
axes[1].grid(True, alpha=0.3, axis='x')

for zone in zone_hourly['zone_type'].unique():
    sub = zone_hourly[zone_hourly['zone_type']==zone].groupby('hour')['avg_fill'].mean()
    axes[2].plot(sub.index, sub.values, label=zone, linewidth=1.5, marker='o', markersize=3)
axes[2].set_title('Predicted Demand by Zone & Hour')
axes[2].set_xlabel('Hour of Day')
axes[2].set_ylabel('Avg Fill Level (%)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/05_rf_demand.png', dpi=150, bbox_inches='tight')
plt.show()

## Part B — Reinforcement Learning Simulation

In [ ]:
# Simple Q-table RL for dynamic bin priority routing
class WasteRouteEnv:
    def __init__(self, n_bins=10):
        self.n_bins = n_bins
        self.fill_levels = np.random.uniform(20, 80, n_bins)
        self.truck_capacity = 500
        self.collected = 0

    def get_state(self):
        return tuple((self.fill_levels > 70).astype(int))

    def step(self, action):
        """Collect bin at index `action`."""
        fill = self.fill_levels[action]
        reward = fill / 100  # Higher fill → higher reward
        if fill < 40:
            reward -= 0.5     # Penalty for collecting nearly-empty bin
        self.fill_levels[action] = np.random.uniform(0, 10)  # Reset bin
        # Simulate other bins filling up
        self.fill_levels += np.random.uniform(0, 3, self.n_bins)
        self.fill_levels = np.clip(self.fill_levels, 0, 100)
        done = self.collected >= self.truck_capacity
        self.collected += fill
        return self.get_state(), reward, done

    def reset(self):
        self.fill_levels = np.random.uniform(20, 80, self.n_bins)
        self.collected = 0
        return self.get_state()

# Q-learning
n_bins = 8
env = WasteRouteEnv(n_bins=n_bins)
Q = {}
alpha, gamma, epsilon = 0.1, 0.9, 1.0
eps_decay = 0.995
rewards_per_ep = []

for ep in range(500):
    state = env.reset()
    total_r = 0
    for _ in range(20):
        if state not in Q:
            Q[state] = np.zeros(n_bins)
        if np.random.rand() < epsilon:
            action = np.random.randint(n_bins)
        else:
            action = np.argmax(Q[state])
        next_state, reward, done = env.step(action)
        if next_state not in Q:
            Q[next_state] = np.zeros(n_bins)
        # Q-update
        Q[state][action] += alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])
        state = next_state
        total_r += reward
        if done: break
    rewards_per_ep.append(total_r)
    epsilon = max(0.05, epsilon * eps_decay)

smoothed = pd.Series(rewards_per_ep).rolling(20).mean()
print(f'RL Training complete. Final avg reward: {np.mean(rewards_per_ep[-50:]):.3f}')
print(f'Initial avg reward: {np.mean(rewards_per_ep[:50]):.3f}')
print(f'Q-table size: {len(Q)} states')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rewards_per_ep, alpha=0.3, color='#9C27B0', label='Episode reward')
ax.plot(smoothed, color='#9C27B0', linewidth=2, label='20-episode rolling avg')
ax.set_title('Reinforcement Learning — Reward Convergence\n(Q-Learning for Dynamic Bin Priority Routing)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/05_rl_convergence.png', dpi=150, bbox_inches='tight')
plt.show()